In [ ]:
import os
import json
import subprocess
import sys
from pathlib import Path
import pandas as pd


## olmOCR

[olmOCR](https://github.com/allenai/olmocr) is a vision-language-model-based OCR pipeline: instead of the region-detect-then-recognize approach used by tools like PaddleOCR, it feeds page images directly into a vision-language model (served locally via [vLLM](https://github.com/vllm-project/vllm)) and asks it to produce clean text/Markdown for the page. This is being run here as a point of comparison against the PaddleOCR pipeline in `multipdf_extraction.ipynb`.

**This notebook requires an NVIDIA GPU** and will not run on Apple Silicon or any CPU-only machine. It is meant to be run **on a GPU droplet** (e.g. a DigitalOcean GPU Droplet), not locally on a Mac — see `olmOCR_DigitalOcean_Guide.md` for the full setup process (creating the droplet, SSH access, and building the Python 3.11 virtual environment with `olmocr[gpu]` and `jupyterlab` installed).

This notebook assumes:

- You're running it **inside that already-activated `olmocr-env` virtual environment** on the droplet (`source ~/olmocr-env/bin/activate`, then launch Jupyter with `--allow-root` since the droplet is accessed as `root`).
- The source PDFs and `manifest.csv` have already been uploaded to the droplet (`scp`, as described in the guide) and live alongside this notebook.
- The manifest uses the **same `filename`/`id` columns** as the PaddleOCR pipeline, so olmOCR's output files end up named to match — useful for lining the two pipelines' results up side by side later.

In [ ]:
#Folder containing the source PDF documents (should already be populated on the droplet)
PDF_FOLDER = Path("pdfs")
#CSV manifest listing which PDFs to process (must have "filename" and "id" columns)
MANIFEST_PATH = "manifest.csv"
#Folder where olmOCR's raw pipeline output (JSONL, logs, etc.) is written
WORKSPACE_DIR = Path("olmocr_workspace")
#Folder where the final per-document .txt files are written
OUTPUT_DIR = Path("extracted_text")


In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
manifest.head()


## Locating the PDFs

Cross-references the manifest's `filename` column against `PDF_FOLDER`, so the same set of documents (and only those documents) get processed as in the PaddleOCR pipeline. Any manifest entry whose file isn't present is reported and skipped rather than silently ignored.

In [ ]:
def check_pdfs(manifest):
    PDF_FOLDER.mkdir(exist_ok=True)
    pdf_files = []
    for _, row in manifest.iterrows():
        pdf_path = PDF_FOLDER / row["filename"]
        if pdf_path.exists():
            pdf_files.append(pdf_path)
        else:
            print(f"Warning: {pdf_path} not found, skipping.")

    print(f"\nFound {len(pdf_files)} of {len(manifest)} manifest PDF(s) in {PDF_FOLDER}/:")
    for f in pdf_files:
        print(f"  - {f.name}")
    if not pdf_files:
        sys.exit("No PDFs found for the manifest entries — check PDF_FOLDER and manifest.csv.")
    return pdf_files

pdf_files = check_pdfs(manifest)


## Running the olmOCR pipeline

Launches `python3 -m olmocr.pipeline`, which starts its own local vLLM server on the droplet's GPU and processes the given PDFs against it (see `python3 -m olmocr.pipeline --help` for the full flag reference — `--model`, `--gpu-memory-utilization`, `--tensor-parallel-size`, etc. configure that local server; `--server`/`--api_key` exist to point at an already-running server elsewhere, not used here).

Output is streamed live rather than captured silently, so progress is visible. On first run you'll see repeating lines like `WARNING - Attempt 42: Please wait for vllm server to become ready...` while the model loads and warms up — this is expected and can take a few minutes; it's only a problem if it continues until the pipeline gives up (~130 attempts).

In [ ]:
def run_pipeline(pdf_files):
    WORKSPACE_DIR.mkdir(exist_ok=True)

    cmd = [
        "python3", "-m", "olmocr.pipeline",
        str(WORKSPACE_DIR),
        "--pdfs", *[str(p) for p in pdf_files],
    ]

    print("Running:", " ".join(cmd))
    print("(this will download model weights on first run — may take a while)\n")

    result = subprocess.run(cmd)
    if result.returncode != 0:
        sys.exit(f"Pipeline exited with code {result.returncode} — check the output above for errors.")

run_pipeline(pdf_files)


## Loading the results

olmOCR writes its output as JSONL files under `olmocr_workspace/results/`, one record per processed page/document. Observed record fields: `id` (a content-hash-based identifier), `text`, `source` (the original PDF path that was passed in), `added`, `created`, `metadata`, `attributes`.

In [ ]:
def load_results():
    results_dir = WORKSPACE_DIR / "results"
    jsonl_files = sorted(results_dir.glob("*.jsonl"))
    print(f"Found {len(jsonl_files)} JSONL result file(s) in {results_dir}/")

    records = []
    for jf in jsonl_files:
        with open(jf) as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))

    if not records:
        sys.exit("No records found — something went wrong upstream. Check olmocr_workspace/ for logs.")

    print(f"Loaded {len(records)} record(s). First record's keys: {list(records[0].keys())}")
    return records

records = load_results()


## Writing per-document text files

For each record, the original PDF's filename is recovered from the `source` field and matched back against the manifest's `filename` → `id` mapping, so the output is named `<id>_output.txt` — the same convention used by the PaddleOCR pipeline. This keeps filenames aligned across both pipelines for an easy side-by-side comparison later. If a record can't be matched back to a manifest row, it falls back to olmOCR's own content-hash `id` and prints a warning.

In [ ]:
def write_text_files(records, manifest):
    OUTPUT_DIR.mkdir(exist_ok=True)

    filename_to_id = dict(zip(manifest["filename"], manifest["id"]))

    for rec in records:
        if "text" not in rec:
            print(f"Warning: record missing 'text' field, skipping: {list(rec.keys())}")
            continue

        source_name = os.path.basename(str(rec.get("source", "")))
        doc_id = filename_to_id.get(source_name)
        if doc_id is None:
            doc_id = rec.get("id", source_name or "unknown")
            print(f"Warning: could not match source '{source_name}' to a manifest entry, using olmOCR id '{doc_id}' instead.")

        out_path = OUTPUT_DIR / f"{doc_id}_output.txt"
        out_path.write_text(rec["text"])
        print(f"Wrote {out_path} ({len(rec['text'])} chars)")

    print(f"\nDone. Text files are in {OUTPUT_DIR.resolve()}/")
    print("From your Mac, pull them down with:")
    print(f"  scp -r root@<droplet-ip>:{OUTPUT_DIR.resolve()} ./extracted_text")

write_text_files(records, manifest)
